# Program do tworzenia map i histogramów nCSP, rysowania RMSD/Lindemanna i generowania filmu na podstawie wygenerowanych obrazków

## Blok pierwszy wczytuje dane z pliku log i dump i ustawia zmienne globalne. Tworzy katalogi na wyniki. 

In [1]:
from ovito.io import import_file
from ovito.io import export_file
from ovito.modifiers import WignerSeitzAnalysisModifier
from ovito.modifiers import CentroSymmetryModifier
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import subprocess
import os
import lammps_logfile


# ----------------------------------------------------------
# ŚCIEŻKI DO PLIKÓW
# ----------------------------------------------------------
filedump = '/home/przemek/Documents/Pd_topnienie/test/grzanie.dump'
logfile_path = '/home/przemek/Documents/Pd_topnienie/test/log.lammps'
mean_r = 2.72  # <- trzeba ustawic recznie na podstawie pdf


# ----------------------------------------------------------
# WCZYTANIE LOGA LAMMPSA
# ----------------------------------------------------------
try:
    log = lammps_logfile.File(logfile_path)
    
    N=2 #number of simulation blocks (which run you want to plot)- counting from 0 
        #trzeba to sprawdzic w pliku log/input - inaczej czasy w wykresach beda zle
    print(f"Data written in log file: {N}: {log.get_keywords(run_num=N)}")
    temps = log.get("Temp", run_num=N)
    times = log.get("Time", run_num=N)
    steps = log.get("Step", run_num=N)
    print("Wczytano dane z log.lammps")
except Exception as e:
    print(f"Nie udało się wczytać log.lammps: {e}")

# ----------------------------------------------------------
# FUNKCJA: znajdź czas i temperaturę odpowiadającą timestepowi
# ----------------------------------------------------------
def find_time_temp_for_step(step):
    if steps is None:
        return None, None
    # znajdź najbliższy timestep (nie zawsze 1:1)
    idx = (np.abs(steps - step)).argmin()
    time_val = times[idx] if times is not None and len(times) > idx else None
    temp_val = temps[idx] if temps is not None and len(temps) > idx else None
    return time_val, temp_val

#def to change particle type from int to name

## !!! tutaj wprowadz swoje nazwy pierwiastkow
def setup_atom_types(frame, data):
    types = data.particles_.particle_types_
    types.type_by_id_(1).name = "Pd" #set your atomic types
    #types.type_by_id_(1).radius = 1.35
    types.type_by_id_(2).name = "Si"
    #types.type_by_id_(2).radius = 1.55
    

pipeline = import_file(filedump)
print("Wczytano dump")
frame_max = pipeline.source.num_frames
print(f"Number of frames in file {frame_max}")

# ----------------------------------------------------------
# Ustal ścieżki do folderów wyjściowych
# ----------------------------------------------------------

# tworzy foldery na wykresy i pliki wynikowe dzialania programu
# domyslnie w katalogu z plikiem dump
base_dir = os.path.dirname(filedump)
hist_dir = os.path.join(base_dir, 'histogramy')
map_dir  = os.path.join(base_dir, 'mapy')

# Utwórz foldery, jeśli nie istnieją
os.makedirs(hist_dir, exist_ok=True)
os.makedirs(map_dir,  exist_ok=True)

Data written in log file: 2: ['Density', 'Enthalpy', 'KinEng', 'Nbuild', 'Ndanger', 'PotEng', 'Press', 'Step', 'Temp', 'Time', 'TotEng', 'Volume']
✅ Wczytano dane z log.lammps
Wczytano dump
Number of frames in file 2201


## Drugi blok rysuje histogramy nCSP i mapy nCSP.

In [ ]:
pipeline.modifiers.clear()
pipeline.modifiers.append(CentroSymmetryModifier())

# ----------------------------------------------------------
# Użytkownik wybiera zakres i krok
# ----------------------------------------------------------
start_frame = int(input("Podaj numer klatki początkowej: "))
end_frame   = int(input("Podaj numer klatki końcowej: "))
step_frame  = int(input("Co ile klatek porównywać (np. 1 = każda kolejna, 5 = co piątą): "))

# Sprawdzenie poprawności
if start_frame < 0 or end_frame >= frame_max:
    raise ValueError(f"Zakres musi mieścić się w przedziale 0 – {frame_max-1}.")
if end_frame <= start_frame:
    raise ValueError("Klatka końcowa musi być większa od początkowej.")
if step_frame <= 0:
    raise ValueError("Krok musi być dodatni (>=1).")
    
for frame in range(start_frame, end_frame, step_frame):
    next_frame = frame + step_frame
    if next_frame > end_frame:
        break

    print(f"\n Przetwarzanie klatek {frame} i {next_frame}...")
    data_frame_init = pipeline.compute(frame=0)
    data_frame0 = pipeline.compute(frame=frame)
    data_frame1 = pipeline.compute(frame=next_frame)
    num_atoms = data_frame1.particles.count

    # --- Oryginalne CSP ---
    csp0 = data_frame0.particles['Centrosymmetry'].array
    csp1 = data_frame1.particles['Centrosymmetry'].array

    # --- Normalizacja CSP ---
    csp0_norm = csp0 / (12*mean_r**2)
    csp1_norm = csp1 / (12*mean_r**2)

    # Timesteps
    timestep_init = data_frame_init.attributes.get("Timestep", 0)
    timestep0 = timestep_init - data_frame0.attributes.get("Timestep", frame)
    timestep1 = data_frame1.attributes.get("Timestep", next_frame) - timestep_init

    # Czas, temperatura, Δt …
    time0, temp0 = find_time_temp_for_step(timestep_init)
    time1, temp1 = find_time_temp_for_step(timestep1)
    #delta_t = (time1 - time0) if (time0 is not None and time1 is not None) else None
    #if delta_t is not None:
    #    print(f"⏱ Odstęp czasowy między klatkami: {delta_t:.3f} ps")

    title_extra = ""
    if time1 is not None: title_extra += f"Time = {time1:.1f} ps"
    if temp1 is not None: title_extra += f", T = {temp1:.1f} K"
    #if delta_t is not None: title_extra += f", Δt = {delta_t:.3f} ps"

    # ----------------------------------------------------------
    # Parametry stałej skali kolorów dla map 2D
    # ----------------------------------------------------------
    COLOR_MIN = 2
    COLOR_MAX = 2000    # ustal sam w zależności od typowego maksimum zliczeń

    # ------------------------------------------------------
    # Histogram nCSP (znormalizowany CSP)
    # ------------------------------------------------------
    plt.figure(figsize=(7,5))
    plt.hist(csp1_norm, bins=100, color='steelblue', edgecolor='black')
    plt.xlabel("nCSP = CSP / (r̄²)")
    plt.ylabel("Number of atoms")
    plt.title(f"Histogram nCSP - klatka {next_frame}{title_extra}")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    hist_path = os.path.join(hist_dir, f"histogram_nCSP_frame{next_frame:03d}.png")
    plt.savefig(hist_path, dpi=300)
    plt.close()

    # ------------------------------------------------------
    # Mapa gęstości nCSP(frame) vs nCSP(next_frame)
    # ------------------------------------------------------
    plt.figure(figsize=(6,6))
    XMIN, XMAX = 0, 2/(mean_r**2)   # zakres skalowany tak jak dane
    YMIN, YMAX = 0, 2/(mean_r**2)
    hist, xedges, yedges = np.histogram2d(csp0_norm, csp1_norm,
                                          bins=200, range=[[XMIN, XMAX], [YMIN, YMAX]])
    hist = np.where(hist == 0, np.nan, hist)
    plt.imshow(hist.T, origin='lower', extent=[XMIN, XMAX, YMIN, YMAX],
               cmap='jet', norm=colors.Normalize(vmin=np.nanmin(hist), vmax=np.nanmax(hist)))
    #plt.imshow(hist.T, origin='lower', extent=[XMIN, XMAX, YMIN, YMAX], 
     #          cmap='jet', norm=colors.Normalize(vmin=COLOR_MIN, vmax=COLOR_MAX))
    plt.plot([XMIN, XMAX], [YMIN, YMAX], 'r-', linewidth=1.2)
    plt.xlabel(f"nCSP (frame {frame})")
    plt.ylabel(f"nCSP (frame {next_frame})")
    plt.title(f"{title_extra}")
    plt.gca().set_facecolor('white')
    #cbar = plt.colorbar()  ## < tworzy slupek ze skala kolorow
    #cbar.set_label("Liczba atomów")
    plt.tight_layout()
    map_path = os.path.join(map_dir, f"nCSP_frame{frame:03d}_vs_{next_frame:03d}.png")
    plt.savefig(map_path, dpi=300)
    plt.close()

    print(f"Zapisano: {hist_path}")
    print(f"Zapisano: {map_path}")
    print(f"Zapisano: {map_path}")

print("\n Zakończono generowanie wszystkich wykresów!")

## Trzeci blok liczy rmsd/lindemmana. Bloki można uruchamiać niezależnie od siebie.

In [ ]:
# === BLOK: OBLICZANIE PARAMETRU LINDEMANN I RMSD ===
import numpy as np
import os
import matplotlib.pyplot as plt
from ovito.modifiers import UnwrapTrajectoriesModifier

# ------------------------------------------------------------------
# Modifier: rozwijamy trajektorie, żeby atomy nie "skakały" przez PBC
# ------------------------------------------------------------------
pipeline.modifiers.append(UnwrapTrajectoriesModifier())

# ------------------------------------------------------------------
# Folder wyjściowy
# ------------------------------------------------------------------
rmsd_dir = os.path.join(base_dir, "rmsd")
os.makedirs(rmsd_dir, exist_ok=True)

# ------------------------------------------------------------------
# Pomocnicze funkcje
# ------------------------------------------------------------------
def displacements_between(data_a, data_b):
    """Zwraca wektor przemieszczeń atomów pomiędzy dwiema klatkami."""
    pos_a = data_a.particles.positions.array
    pos_b = data_b.particles.positions.array
    ids_a = data_a.particles["Particle Identifier"].array
    ids_b = data_b.particles["Particle Identifier"].array

    # znajdź wspólne atomy (te same identyfikatory)
    common_ids, idx_a, idx_b = np.intersect1d(ids_a, ids_b, return_indices=True)
    if len(common_ids) == 0:
        raise ValueError("Brak wspólnych ID atomów między klatkami!")

    pos_a_matched = pos_a[idx_a]
    pos_b_matched = pos_b[idx_b]

    # różnice pozycji dla każdego atomu
    diff_vectors = pos_b_matched - pos_a_matched
    return diff_vectors

def rmsd_and_lindemann(diff_vectors, mean_r):
    """Oblicza RMSD i parametr Lindemanna."""
    # RMSD = sqrt( 1/N * sum_i |r_i(t+Δt) - r_i(t)|² )
    squared_displacements = np.sum(diff_vectors**2, axis=1)
    rmsd = np.sqrt(np.mean(squared_displacements))
    lindemann = rmsd / mean_r
    return rmsd, lindemann

# ------------------------------------------------------------------
# Zakres klatek i krok iteracji
# ------------------------------------------------------------------
print(f"\n W pliku dostępnych jest {frame_max} klatek (0–{frame_max-1}).")
start_frame = int(input("Podaj numer klatki początkowej: "))
end_frame   = int(input("Podaj numer klatki końcowej: "))
step_frame  = int(input("Co ile klatek liczyć RMSD (np. 1 = co każdą, 5 = co piątą): "))
print("W tym miejscu muszę pomyśleć - poczekaj!")

if start_frame < 0 or end_frame >= frame_max:
    raise ValueError(f"Zakres musi mieścić się w przedziale 0 – {frame_max-1}.")
if end_frame <= start_frame:
    raise ValueError("Klatka końcowa musi być większa od początkowej.")
if step_frame <= 0:
    raise ValueError("Krok iteracji musi być dodatni (>= 1).")

# ------------------------------------------------------------------
# Ustawienia: porównujemy każdą kolejną klatkę względem pierwszej
# ------------------------------------------------------------------
### TUTAJ decydujesz ktora klatka ma byc referencyjna, pierwsza czy poprzedzajaca
# True - wzgledem 1; False - wzgledem poprzedzajacej

compare_to_first = False

data_ref = pipeline.compute(frame=start_frame) if compare_to_first else None

# ------------------------------------------------------------------
# Wyznacz czas początkowy (do przesunięcia do zera)
# ------------------------------------------------------------------
if steps is not None:
    if compare_to_first:
        # mamy data_ref (start_frame)
        timestep0 = data_ref.attributes.get("Timestep", start_frame)
    else:
        # nie ma data_ref, więc pobieramy pierwszą klatkę z zakresu
        data_start = pipeline.compute(frame=start_frame)
        timestep0 = data_start.attributes.get("Timestep", start_frame)

    time0, _ = find_time_temp_for_step(timestep0)
    if time0 is None:
        time0 = 0.0
else:
    time0 = 0.0

# ------------------------------------------------------------------
# Główna pętla
# ------------------------------------------------------------------
wyniki = []
for frame in range(start_frame, end_frame, step_frame):
    next_frame = frame + step_frame
    if next_frame > end_frame:
        break
    print(f"\n Przetwarzanie ramek {frame} i {next_frame} ...")

    data_next = pipeline.compute(frame=next_frame)

    if compare_to_first:
        diff_vectors = displacements_between(data_ref, data_next)
    else:
        data_curr = pipeline.compute(frame=frame)
        diff_vectors = displacements_between(data_curr, data_next)

    rmsd, lindemann = rmsd_and_lindemann(diff_vectors, mean_r)
    num_atoms = data_next.particles.count

    # zapis pojedynczego rozkładu RMSD
    outpath = os.path.join(rmsd_dir, f"rmsd_frame{frame:04d}_to_{next_frame:04d}.dat")
    np.savetxt(outpath, np.linalg.norm(diff_vectors, axis=1), fmt="%.6f")

    # czas z loga – przesunięty do zera
    timestep = data_next.attributes.get("Timestep", next_frame)
    t_val, _ = find_time_temp_for_step(timestep)
    t_rel = (t_val - time0) if t_val is not None else np.nan

    wyniki.append([frame, next_frame, t_rel, rmsd, lindemann, num_atoms])
    print(f"➡ Frame {frame:04d}–{next_frame:04d}: RMSD={rmsd:.4f}, Lindemann={lindemann:.4f}, czas={t_rel:.3f} ps")

# ------------------------------------------------------------------
# Zbiorczy plik wyników
# ------------------------------------------------------------------
wyniki = np.array(wyniki)
header = "frame1\tframe2\ttime(ps)\tRMSD\tLindemann\tNumAtoms"
summary_path = os.path.join(rmsd_dir, "rmsd_lindemann_summary.dat")
np.savetxt(summary_path, wyniki, fmt="%.6f", header=header)
print(f"\n Zbiorczy plik zapisany jako: {summary_path}")

# ------------------------------------------------------------------
# Wykresy RMSD i Lindemanna
# ------------------------------------------------------------------
times_plot = wyniki[:, 2]
rmsd_vals = wyniki[:, 3]
lindemann_vals = wyniki[:, 4]

plt.figure(figsize=(7,5))
plt.plot(times_plot, rmsd_vals, marker='o', color='tab:blue')
plt.xlabel("Czas relatywny [ps]")
plt.ylabel("RMSD")
plt.grid(alpha=0.3)
plt.title("RMSD vs czas (relatywny, t₀ = 0 ps)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,5))
plt.plot(times_plot, lindemann_vals, marker='o', color='tab:red')
plt.xlabel("Czas relatywny [ps]")
plt.ylabel("Parametr Lindemanna")
plt.grid(alpha=0.3)
plt.title("Lindemann vs czas (relatywny, t₀ = 0 ps)")
plt.tight_layout()
plt.show()

## Ten blok tworzy filmik. Domyślnie z obrazkami z map nCSP.

In [23]:
from moviepy import ImageSequenceClip
import glob
import os
import re

input_dir = map_dir
output_movie = os.path.join(base_dir, "mapy_nCSP_moviepy.mp4")

image_files = sorted(
    glob.glob(os.path.join(input_dir, "nCSP_frame*_vs_*.png")),
    key=lambda f: int(re.search(r'nCSP_frame(\d+)_vs_', f).group(1))
)

fps = int(input("\n🎞 Podaj liczbę klatek na sekundę: "))
quality = input("Podaj jakość wyjścia (np. 'medium', 'high', 'low'): ")

clip = ImageSequenceClip(image_files, fps=fps)
clip.write_videofile(output_movie, codec="libx264", preset=quality, threads=4)

print(f"✅ Film zapisano jako: {output_movie}")


🎞 Podaj liczbę klatek na sekundę:  10
Podaj jakość wyjścia (np. 'medium', 'high', 'low'):  medium


MoviePy - Building video /home/przemek/Documents/Pd_topnienie/test/mapy_nCSP_moviepy.mp4.
MoviePy - Writing video /home/przemek/Documents/Pd_topnienie/test/mapy_nCSP_moviepy.mp4



MoviePy - Done !
MoviePy - video ready /home/przemek/Documents/Pd_topnienie/test/mapy_nCSP_moviepy.mp4
✅ Film zapisano jako: /home/przemek/Documents/Pd_topnienie/test/mapy_nCSP_moviepy.mp4
